# Setup no Colab
Runtime: **Ambiente de execução → Alterar tipo → GPU**. Rode as células em ordem.
Dados e resultados ficam no Drive (`MyDrive/mestrado-epi`); o código vem do repositório.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys; print(sys.version)

/bin/bash: line 1: nvidia-smi: command not found
3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]


## 1. Drive (persistência de data/ e runs/)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/mestrado-epi'
os.makedirs(f'{DRIVE}/data', exist_ok=True); os.makedirs(f'{DRIVE}/runs', exist_ok=True)

## 2. Código
Preencha `REPO_URL` (o repositório precisa estar num remoto, ex. GitHub; repo privado exige token em Secrets do Colab).

In [ ]:
REPO_URL = ''  # TODO: https://github.com/<usuario>/<repo>.git
REPO = '/content/projeto-mestrado'
assert REPO_URL, 'defina REPO_URL'
if os.path.exists(REPO): !git -C {REPO} pull
else: !git clone {REPO_URL} {REPO}
%cd {REPO}

## 3. Dependências
O Colab já traz PyTorch/CUDA; instala só o que falta. Usa `src/` direto (sem `pip install -e`).

In [ ]:
!pip install -q ultralytics pyyaml pytest
import sys
sys.path.insert(0, f'{REPO}/src')
# data/ e runs/ apontam para o Drive
!rm -rf data runs && ln -s {DRIVE}/data data && ln -s {DRIVE}/runs runs

## 4. Verificação

In [ ]:
import torch, ultralytics
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('ultralytics', ultralytics.__version__)
!python -m pytest -q

## 5. Treino (modelo)
Template; `project='runs'` grava no Drive. Ajuste quando o dataset/YAML existir.

In [ ]:
# from ultralytics import YOLO
# YOLO('yolov8s.pt').train(data='configs/ppe.yaml', epochs=50, imgsz=640, batch=32,
#                           seed=0, project='runs', name='e1_baseline')